In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore, MaximumLikelihoodEstimator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
df = pd.read_parquet("../classify_data.parquet").reset_index()

In [2]:
class_map = {
    '0': 0,
    '1-3': 1,
    '4-7': 2,
    '8-14': 3,
    '15-30': 4,
    '31-60': 5
}
df['resolution_class_int'] = df['resolution_class'].map(class_map)

In [ ]:
selected_columns = [
        'Agency', 
        'Complaint Type', 
        'Location Type', 
        'City', 
        'Borough', 
        'Incident Zip',
        'Open Data Channel Type', 
        'hour_of_day', 
        'day_of_week', 
        'month',
        'is_weekend',
        "open_requests_past_24h",
        'resolution_class_int'
    ]

df_bn = df[selected_columns].copy()

df_bn['time_of_day'] = pd.cut(
        df_bn['hour_of_day'], 
        bins=[-1, 6, 12, 18, 24], 
        labels=['Night', 'Morning', 'Afternoon', 'Evening']
    )
    # Drop original hour
df_bn = df_bn.drop(columns=['hour_of_day'])

# Handle missing values (BBNs generally don't like NaNs during structure learning)
df_bn.dropna(inplace=True)

# ---------------------------------------------------------
# 3. Structure Learning (DAG)
# ---------------------------------------------------------
# We learn the structure ONCE using a sample
print("2. Learning Network Structure (DAG)...")

# Sample for structure learning (10k rows is a good balance for speed/accuracy)
struct_data = df_bn.sample(n=10000, random_state=42)

# Hill Climb Search with BIC Score
hc = HillClimbSearch(struct_data)
best_model_structure = hc.estimate(scoring_method=BicScore(struct_data))

print("   Structure learned. Edges found:")
print(best_model_structure.edges())

# ---------------------------------------------------------
# 4. Train / Test Split (Simple Split)
# ---------------------------------------------------------
print("\n3. Splitting Data (80/20 Stratified Split)...")

# We split the dataframe into Train and Test
# Stratify ensures class balance is maintained in both sets
train_data, test_data = train_test_split(
    df_bn, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_bn['resolution_class_int']
)

print(f"   Training samples: {len(train_data)}")
print(f"   Testing samples:  {len(test_data)}")

# ---------------------------------------------------------
# 5. Model Training & Parameter Learning
# ---------------------------------------------------------
print("\n4. Training Model on Training Set...")

# Define Model with the learned structure
model = BayesianNetwork(best_model_structure.edges())



In [ ]:
model.fit(train_data, estimator=MaximumLikelihoodEstimator)
    
# ---------------------------------------------------------
# 6. Prediction & Evaluation
# ---------------------------------------------------------
print("\n5. Predicting on Test Set...")

# Prepare Test Data (Drop target)
X_test = test_data.drop(columns=['resolution_class_int'])
y_true = test_data['resolution_class_int']

# Predict
# Note: pgmpy predict can be slow on large datasets. 
# If it hangs, consider sampling the test set for the demo.
y_pred_df = model.predict(X_test)
y_pred = y_pred_df['resolution_class_int']



In [ ]:
# Score
acc = accuracy_score(y_true, y_pred)
print(f"\nTest Accuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
print("\n6. Saving Network Graph to '../results/bayes_network_graph.png'...")
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(model, k=0.5, iterations=20)
nx.draw(
    model, pos, 
    with_labels=True, 
    node_size=3000, 
    node_color="lightblue", 
    font_size=10, 
    font_weight="bold", 
    arrowsize=20,
    edge_color="gray"
)
plt.title("Learned Bayesian Network Structure")
plt.savefig("../results/bayes_network_graph.png")
print("   Graph saved.")